# Project: Multi-Agent Research System

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will build a multi-agent research system with a router, web researcher, summarizer, and report writer.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph langchain-mcp-adapters

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define the Search Tool

Create a web search tool that the researcher agent will use to gather information.

In [ ]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for information on a topic."""
    results = {
        "machine learning trends 2025": (
            "[Source: MIT Tech Review] Multimodal models dominate enterprise AI adoption. "
            "Fine-tuning smaller models outperforms prompting larger ones for specific tasks. "
            "Agent frameworks emerge as the primary way to build AI applications."
        ),
        "climate change solutions": (
            "[Source: Nature] Carbon capture technology scales to industrial levels. "
            "Solar energy costs drop below fossil fuels globally. "
            "Reforestation programs show measurable impact on CO2 levels."
        ),
        "quantum computing": (
            "[Source: IEEE] Quantum error correction reaches practical thresholds. "
            "Hybrid quantum-classical algorithms show promise in optimization. "
            "Major cloud providers offer quantum computing as a service."
        ),
    }
    for key, value in results.items():
        if any(word in query.lower() for word in key.split()):
            return value
    return f"Search results for '{query}': General information found. Consider refining your query."

print(f"Tool: {web_search.name}")
print(f"Description: {web_search.description}")

## 4. Create Specialist Agents

Build a web researcher, summarizer, and report writer — each with a focused role.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4o-mini", model_provider="openai")

web_researcher = create_react_agent(
    model=model,
    tools=[web_search],
    name="web_researcher",
    prompt=(
        "You are a web researcher. Use the web_search tool to find information. "
        "Return raw findings with source attributions. Do not summarize."
    ),
)

summarizer = create_react_agent(
    model=model,
    tools=[],
    name="summarizer",
    prompt=(
        "You are a summarizer. Take lengthy text and condense it into "
        "3-5 key bullet points. Preserve source citations."
    ),
)

report_writer = create_react_agent(
    model=model,
    tools=[],
    name="report_writer",
    prompt=(
        "You are a report writer. Take research findings and produce a structured report "
        "with sections: Overview, Key Findings, and Conclusion. Include citations."
    ),
)

print("Specialist agents created: web_researcher, summarizer, report_writer")

## 5. Build the Router Agent

Create a router that dispatches to the right specialist using handoffs.

In [ ]:
router = create_react_agent(
    model=model,
    tools=[],
    name="router",
    prompt=(
        "You are a research coordinator. Analyze the user's request and route to the right specialist:\n"
        "- web_researcher: when the user needs new information gathered\n"
        "- summarizer: when the user has text that needs condensing\n"
        "- report_writer: when the user has findings ready for a final report\n"
        "For a full research request, start with web_researcher."
    ),
    handoffs=["web_researcher", "summarizer", "report_writer"],
)

print("Router agent created with handoffs to all specialists.")

## 6. Run a Research Query

Send a research request through the router.

In [ ]:
from langchain_core.messages import HumanMessage

result = router.invoke({
    "messages": [HumanMessage(content="Research the latest trends in machine learning")]
})
print(result["messages"][-1].content)

## 7. Structured Report Output

Define a Pydantic model for structured report output.

In [ ]:
from pydantic import BaseModel, Field

class ReportOutput(BaseModel):
    title: str = Field(description="Report title")
    overview: str = Field(description="Brief overview of the research topic")
    key_findings: list[str] = Field(description="List of key findings with citations")
    conclusion: str = Field(description="Summary conclusion")
    sources: list[str] = Field(description="List of sources cited")

structured_report_writer = create_react_agent(
    model=model,
    tools=[],
    name="structured_report_writer",
    prompt=(
        "You are a report writer. Produce a structured research report. "
        "Include an overview, key findings with citations, conclusion, and sources list."
    ),
    response_format=ReportOutput,
)

result = structured_report_writer.invoke({
    "messages": [HumanMessage(content=(
        "Write a report based on these findings: "
        "[MIT Tech Review] Multimodal models dominate. Fine-tuning outperforms prompting. "
        "Agent frameworks emerge as primary AI dev pattern."
    ))]
})
print(result["messages"][-1].content)

## 8. Streaming Progress

Stream agent events to show real-time progress through the pipeline.

In [ ]:
async for event in router.astream_events(
    {"messages": [HumanMessage(content="Research climate change solutions")]},
    version="v2",
):
    if event["event"] == "on_chat_model_stream":
        chunk = event["data"]["chunk"]
        if chunk.content:
            print(chunk.content, end="", flush=True)
    elif event["event"] == "on_tool_start":
        print(f"\n[Using tool: {event['name']}]")

print()

## 9. Multiple Research Queries

Run several research queries through the system.

In [ ]:
queries = [
    "Research quantum computing breakthroughs",
    "Research climate change solutions",
]

for query in queries:
    result = router.invoke({
        "messages": [HumanMessage(content=query)]
    })
    print(f"Query: {query}")
    print(f"Result: {result['messages'][-1].content[:200]}...\n")

## Summary

- A router agent dispatches to specialist agents using the `handoffs` parameter
- Specialists (researcher, summarizer, report writer) each have focused prompts and tools
- `response_format` with a Pydantic model produces structured output
- `astream_events` streams real-time progress through the pipeline
- The pattern scales by adding more specialists without modifying the router